In [1]:
# ===== [KAGGLE] bootstrap: recreate the Colab layout =====
# The paper notebooks expect /content/drive/MyDrive/bsm_paper_replication and
# /content/BSM_Project_2. bsm_env builds both from the attached dataset; the
# pipeline itself (images, augmentation, splits, training) is unchanged.
import pathlib, sys
_code = pathlib.Path("/kaggle/input/bsm-paper-2-kaggle/code")
if not _code.is_dir():
    _hits = list(pathlib.Path("/kaggle/input").rglob("bsm_env.py"))
    _code = _hits[0].parent if _hits else _code
if str(_code) not in sys.path:
    sys.path.insert(0, str(_code))
import bsm_env
BSM_RAW_SRC = bsm_env.setup()


[bsm_env] using existing /content
[bsm_env] mode=existing  project=/content/drive/MyDrive/bsm_paper_replication
[bsm_env] raw images=367  originals=367
[bsm_env] models.py: ok
[bsm_env] common.py: ok
[bsm_env] torch 2.11.0+cu128 cuda=True Tesla T4
[bsm_env] disk /kaggle/working free=20.9 GB


# BSM Paper Replication — Notebook 08: VGG16 (paper-faithful, real run)

60 epochs, full-batch grad-accum (micro 32, one optim step/epoch), Adam 1e-3, wd 0,
best-validation checkpoint, evaluated on the full 15 % test split.

**Param count note:** `vgg16(head='paper')` = 13 conv layers (2,2,3,3,3 blocks; 64,128,256,512,512) +
FC(4096) -> FC(4096) -> FC(6), dropout 0.5, no batch norm = **138,357,544** params
(DEVIATIONS row 12: the paper's Table 4 is incomplete, the prose describes standard VGG16).


In [2]:
# setup
import os, sys
# [kaggle] Colab Drive mount removed - bsm_env recreates the folder tree
PROJECT = "/content/drive/MyDrive/bsm_paper_replication"
if PROJECT not in sys.path: sys.path.insert(0, PROJECT)
import torch, random, numpy as np
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("gpu:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")


gpu: True Tesla T4


## Real run: VGG16, 60 epochs, full-batch grad-accum (micro 32, one optim step/epoch), Adam 1e-3, best-val checkpoint

In [3]:
# ===== REAL TRAINING: VGG16 (60 epochs, paper-faithful) =====
import os, sys, time, json, pathlib
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader

PROJECT = "/content/drive/MyDrive/bsm_paper_replication"
VER = pathlib.Path(PROJECT) / "versions"
CKPT = pathlib.Path(PROJECT) / "checkpoints" / "vgg16"
RES  = pathlib.Path(PROJECT) / "results"
CKPT.mkdir(parents=True, exist_ok=True)
RES.mkdir(parents=True, exist_ok=True)
if PROJECT not in sys.path: sys.path.insert(0, PROJECT)

from models import vgg16
import common

def load_xy(prefix):
    X = np.load(VER / f"X_{prefix}.npy")
    y = np.load(VER / f"y_{prefix}.npy")
    Xt = torch.from_numpy(X.transpose(0, 3, 1, 2)).contiguous()  # NHWC -> NCHW
    yt = torch.from_numpy(y.astype(np.int64))
    return Xt, yt

Xtr, ytr = load_xy("train")
Xva, yva = load_xy("val")
print("train:", Xtr.shape, "val:", Xva.shape, "| dtype", Xtr.dtype, "range", float(Xtr.min()), float(Xtr.max()))

MICRO = 32
train_loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=MICRO, shuffle=False)
val_loader   = DataLoader(TensorDataset(Xva, yva), batch_size=MICRO, shuffle=False)

torch.manual_seed(42)
model = vgg16().cuda()
n_params = sum(p.numel() for p in model.parameters())
print("vgg16 params:", f"{n_params:,}")

EPOCHS = 60
t0 = time.time()
hist = common.train_model(
    model, train_loader, val_loader, torch.device("cuda"),
    epochs=EPOCHS, full_batch=True,
    ckpt_dir=CKPT, history_path=RES / "history_vgg16.json",
    tag="vgg16", lr=1e-3, resume=True,
)
print(f"TRAINING DONE in {(time.time()-t0)/60:.1f} min")
print("best epoch:", hist.get("best_epoch"), "best val:", hist.get("best_val_acc"))

tr_acc = hist.get("train_acc") or []
if len(tr_acc) >= 30 and tr_acc[29] < 0.30:
    print("WARNING: train acc at epoch 30 =", tr_acc[29], "< 0.30 (spec stop-check)")


train: torch.Size([5651, 3, 224, 224]) val: torch.Size([1211, 3, 224, 224]) | dtype torch.float32 range 0.0 1.0
vgg16 params: 134,285,126
vgg16 epoch 01/60  train_loss 1.7920 acc 0.1686  val_loss 179.1609 acc 0.1800  (51.1s)
vgg16 epoch 02/60  train_loss 183.5383 acc 0.1798  val_loss 179.1609 acc 0.1800  (47.2s)
vgg16 epoch 03/60  train_loss 183.8877 acc 0.1798  val_loss 1.8262 acc 0.1635  (48.4s)
vgg16 epoch 04/60  train_loss 1.8257 acc 0.1635  val_loss 1.7965 acc 0.1685  (47.6s)
vgg16 epoch 05/60  train_loss 1.7963 acc 0.1672  val_loss 1.7936 acc 0.1685  (46.5s)
vgg16 epoch 06/60  train_loss 1.7935 acc 0.1690  val_loss 1.7895 acc 0.1685  (46.1s)
vgg16 epoch 07/60  train_loss 1.7896 acc 0.1690  val_loss 1.8089 acc 0.1685  (46.0s)
vgg16 epoch 08/60  train_loss 1.8113 acc 0.1692  val_loss 1.7897 acc 0.1916  (46.3s)
vgg16 epoch 09/60  train_loss 1.7897 acc 0.1862  val_loss 1.7920 acc 0.1635  (45.7s)
vgg16 epoch 10/60  train_loss 1.7920 acc 0.1635  val_loss 1.7925 acc 0.1635  (45.7s)
vgg1

### Re-engineering round (DEVIATIONS row 14)

If the train-vs-validation gap at the best-validation epoch exceeds 0.15, ONE
extra round is trained from a fresh model with lr 5e-4 and weight decay 1e-4, and
the round with the higher validation accuracy is the one evaluated on the test
split. The threshold and the round-1 hyper-parameters are our documented choice
(the paper does not define them).

In [4]:
# --- RE-ENGINEERING ROUND (spec: re-engineering if train - val gap > 0.15) ---
# Paper's Fig. 5 "model re-engineering" loop. The 0.15 threshold and the
# round-1 hyper-parameters (lr 5e-4, weight decay 1e-4) are OUR documented
# choice -- the paper does not define them (DEVIATIONS row 14).
import json as _json
import numpy as _np
import torch as _torch

RE_ENGINEER_GAP = 0.15
RE_ENGINEER_LR, RE_ENGINEER_WD = 5e-4, 1e-4
RE_ENGINEER_EPOCHS = 60

def _round_stats(tag, history_file):
    h = _json.load(open(history_file))
    b = int(_np.argmax(h["val_acc"])) + 1
    return {"tag": tag, "best_epoch": b,
            "best_val": float(h["val_acc"][b - 1]),
            "train_at_best": float(h["train_acc"][b - 1]),
            "train_seconds": float(_np.sum(h.get("epoch_time_s", []))),
            "gap": float(h["train_acc"][b - 1] - h["val_acc"][b - 1])}

ROUNDS = [_round_stats("vgg16", RES / "history_vgg16.json")]
print("round 0:", ROUNDS[0])

if ROUNDS[0]["gap"] > RE_ENGINEER_GAP:
    print(f"gap {ROUNDS[0]['gap']:+.4f} > {RE_ENGINEER_GAP} -> RE-ENGINEERING ROUND 1 "
          f"(fresh model, lr {RE_ENGINEER_LR}, wd {RE_ENGINEER_WD}, {RE_ENGINEER_EPOCHS} epochs)")
    _torch.manual_seed(SEED)
    model_re1 = vgg16().cuda()
    common.train_model(
        model_re1, train_loader, val_loader, _torch.device("cuda"),
        epochs=RE_ENGINEER_EPOCHS, full_batch=True,
        ckpt_dir=CKPT, history_path=RES / "history_vgg16_re1.json",
        tag="vgg16_re1", lr=RE_ENGINEER_LR, wd=RE_ENGINEER_WD, resume=True,
    )
    ROUNDS.append(_round_stats("vgg16_re1", RES / "history_vgg16_re1.json"))
    print("round 1:", ROUNDS[1])
else:
    print(f"gap {ROUNDS[0]['gap']:+.4f} <= {RE_ENGINEER_GAP} -> no re-engineering round needed")

SELECTED_ROUND = max(ROUNDS, key=lambda r: r["best_val"])
print("SELECTED ROUND (higher validation accuracy):", SELECTED_ROUND)
with open(RES / "selected_round_vgg16.json", "w") as f:
    _json.dump({"rounds": ROUNDS, "selected": SELECTED_ROUND}, f, indent=2)


round 0: {'tag': 'vgg16', 'best_epoch': 43, 'best_val': 0.27167630057803466, 'train_at_best': 0.1787294284197487, 'train_seconds': 2713.100614309311, 'gap': -0.09294687215828595}
gap -0.0929 <= 0.15 -> no re-engineering round needed
SELECTED ROUND (higher validation accuracy): {'tag': 'vgg16', 'best_epoch': 43, 'best_val': 0.27167630057803466, 'train_at_best': 0.1787294284197487, 'train_seconds': 2713.100614309311, 'gap': -0.09294687215828595}


In [5]:
# ===== EVAL VGG16 (best-val ckpt) on TEST SPLIT =====
import os, sys, json, pathlib
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report

PROJECT = "/content/drive/MyDrive/bsm_paper_replication"
VER = pathlib.Path(PROJECT) / "versions"
CKPT = pathlib.Path(PROJECT) / "checkpoints" / "vgg16"
RES  = pathlib.Path(PROJECT) / "results"
if PROJECT not in sys.path: sys.path.insert(0, PROJECT)
from models import vgg16
import common

CLASSES = ["Ilmenite","Rutile","Zircon","Garnet","Monazite","Sillimanite"]

def load_xy(prefix):
    X = np.load(VER / f"X_{prefix}.npy")
    y = np.load(VER / f"y_{prefix}.npy")
    return torch.from_numpy(X.transpose(0,3,1,2)).contiguous(), torch.from_numpy(y.astype(np.int64))

Xte, yte = load_xy("test")
print("test:", Xte.shape)
test_loader = DataLoader(TensorDataset(Xte, yte), batch_size=32, shuffle=False)

device = torch.device("cuda")
model = vgg16().to(device)
print("evaluating selected round:", SELECTED_ROUND)
ck = torch.load(CKPT / f"{SELECTED_ROUND['tag']}_best.pt", map_location=device, weights_only=True)
model.load_state_dict(ck)
_ck = {"epoch": SELECTED_ROUND["best_epoch"]}
model.eval()

all_preds, all_true = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        logits = model(xb.to(device))
        all_preds.append(logits.argmax(1).cpu().numpy())
        all_true.append(yb.numpy())
y_pred = np.concatenate(all_preds); y_true = np.concatenate(all_true)

acc = accuracy_score(y_true, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
cm = confusion_matrix(y_true, y_pred, labels=range(6))
spec_list = []
for i in range(6):
    tn = cm.sum() - cm[i,:].sum() - cm[:,i].sum() + cm[i,i]
    fp = cm[:,i].sum() - cm[i,i]
    spec_list.append(tn/(tn+fp) if (tn+fp)>0 else 0.0)
spec = float(np.mean(spec_list))

print(f"\n=== VGG16 TEST SPLIT (best epoch {_ck['epoch']}) ===")
print(f"accuracy {acc:.4f} | macro precision {prec:.4f} | macro recall {rec:.4f} | macro F1 {f1:.4f} | macro specificity {spec:.4f}")
print("\nconfusion matrix (rows=true):")
print(cm)
print("\nper-class:")
print(classification_report(y_true, y_pred, target_names=CLASSES, digits=4, zero_division=0))

results = {
  "model": "vgg16", "notebook": "08_vgg16",
  "best_epoch": int(_ck.get("epoch", -1)),
  "selected_round": SELECTED_ROUND,
  "re_engineering": ROUNDS,
  "train_minutes": round(sum(r.get("train_seconds", 0.0) for r in ROUNDS) / 60, 2),
  "test_split": {"accuracy": float(acc), "macro_precision": float(prec), "macro_recall": float(rec),
                 "macro_f1": float(f1), "macro_specificity": spec,
                 "confusion_matrix": cm.tolist(), "n_test": int(len(y_true))},
  "paper_reported_table9": {"accuracy": 0.83, "precision": 0.89, "recall": 0.83, "note": "PAPER numbers (Table 9), for comparison only"},
}
with open(RES / "metrics_vgg16.json", "w") as f:
    json.dump(results, f, indent=2)
print("\nsaved", RES / "metrics_vgg16.json")
print("EVAL DONE")


test: torch.Size([1212, 3, 224, 224])
evaluating selected round: {'tag': 'vgg16', 'best_epoch': 43, 'best_val': 0.27167630057803466, 'train_at_best': 0.1787294284197487, 'train_seconds': 2713.100614309311, 'gap': -0.09294687215828595}

=== VGG16 TEST SPLIT (best epoch 43) ===
accuracy 0.2607 | macro precision 0.1325 | macro recall 0.2575 | macro F1 0.1644 | macro specificity 0.8518

confusion matrix (rows=true):
[[  0 115   0  27  63   0]
 [  0 110   0  40  68   0]
 [  0  82   0  25  88   0]
 [  0  26   0  46 126   0]
 [  0  17   0  21 160   0]
 [  0  54   0  10 134   0]]

per-class:
              precision    recall  f1-score   support

    Ilmenite     0.0000    0.0000    0.0000       205
      Rutile     0.2723    0.5046    0.3537       218
      Zircon     0.0000    0.0000    0.0000       195
      Garnet     0.2722    0.2323    0.2507       198
    Monazite     0.2504    0.8081    0.3823       198
 Sillimanite     0.0000    0.0000    0.0000       198

    accuracy                 

In [6]:
# persist notebook copy
import glob, os, shutil
PROJECT = "/content/drive/MyDrive/bsm_paper_replication"
os.makedirs(os.path.join(PROJECT,"notebooks"), exist_ok=True)
cands = glob.glob("/content/drive/MyDrive/Colab Notebooks/*.ipynb")
if cands:
    newest = max(cands, key=os.path.getmtime)
    shutil.copy(newest, os.path.join(PROJECT,"notebooks","08_vgg16.ipynb"))
    print("copied ->", os.path.join(PROJECT,"notebooks","08_vgg16.ipynb"))
